## Cell 1 — Imports & Logging

In [0]:
import logging
import requests
import pandas as pd


from datetime import datetime, timezone, date
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s"
)
logger = logging.getLogger("stl_noaa_ingest")

print("✓ Imports loaded")

## Cell 2 — Configuration

In [0]:
# ── API token ─────────────────────────────────────────────────
# Store this as a Databricks secret in production:
#   databricks secrets put --scope noaa --key api_token
# Then retrieve with:
#   dbutils.secrets.get(scope="noaa", key="api_token")
# For now we set it directly — rotate the token if it gets exposed.
NOAA_TOKEN = "oYEmDmMvIIgstVGZRmjkDLlMFWMGSLlI"

# ── Output path ───────────────────────────────────────────────
RAW_OUTPUT_PATH = "/Volumes/workspace/default/raw/weather"

# ── Date range — match city crime data window ─────────────────
START_DATE = "2024-01-01"
END_DATE   = date.today().isoformat()

# ── St. Louis weather stations ────────────────────────────────
# Lambert Airport is the primary station — longest record,
# most complete data, official NWS station for STL metro.
# Science Center is a secondary station for gap-filling.
STATIONS = {
    "lambert":       "GHCND:USW00013994",   # STL Lambert Airport
    "science_center": "GHCND:USC00237452",  # STL Science Center
}

# ── Data types we want ────────────────────────────────────────
# TMAX  — max temperature (tenths of °C)
# TMIN  — min temperature (tenths of °C)
# PRCP  — precipitation (tenths of mm)
# SNOW  — snowfall (mm)
# SNWD  — snow depth (mm)
# AWND  — average wind speed (tenths of m/s)
DATA_TYPES = ["TMAX", "TMIN", "PRCP", "SNOW", "SNWD", "AWND"]

print(f"✓ Config set")
print(f"  Date range:  {START_DATE} → {END_DATE}")
print(f"  Stations:    {list(STATIONS.keys())}")
print(f"  Data types:  {DATA_TYPES}")
print(f"  Output path: {RAW_OUTPUT_PATH}")

## Cell 3 — API Base URL & Headers

In [0]:
# New NOAA NCEI Access Data Service (replaces deprecated CDO v2)
# Documentation: https://www.ncei.noaa.gov/support/access-data-service-api-user-documentation
NOAA_BASE_URL = "https://www.ncei.noaa.gov/access/services/data/v1"

# The token goes in the request header, not as a query parameter
NOAA_HEADERS = {
    "token": NOAA_TOKEN,
    "User-Agent": "STL-Neighborhood-Pipeline/1.0",
}

print(f"✓ API base URL: {NOAA_BASE_URL}")

## Cell 4 — Canonical Schema

In [0]:
CANONICAL_SCHEMA: dict[str, object] = {
    "station_id":    StringType(),
    "date":          StringType(),
    "year":          IntegerType(),
    "month":         IntegerType(),
    # ── Temperature (both units) ──────────────────────────────
    # Celsius comes directly from the API (metric mode).
    # Fahrenheit is derived: F = (C × 9/5) + 32
    "tmax_c":        DoubleType(),    # max temp °C
    "tmin_c":        DoubleType(),    # min temp °C
    "tmax_f":        DoubleType(),    # max temp °F — derived
    "tmin_f":        DoubleType(),    # min temp °F — derived
    # ── Precipitation & snow ──────────────────────────────────
    "prcp":          DoubleType(),    # precipitation mm
    "snow":          DoubleType(),    # snowfall mm
    "snwd":          DoubleType(),    # snow depth mm
    # ── Wind ─────────────────────────────────────────────────
    "awnd":          DoubleType(),    # avg wind speed m/s
}

print(f"✓ Canonical schema: {len(CANONICAL_SCHEMA)} fields")
for field, dtype in CANONICAL_SCHEMA.items():
    print(f"  {field:15s} {dtype}")

## Cell 5 — Fetch Helper

In [0]:
# Page size — NOAA returns max 1000 records per request.
# For daily data over 2 years that's ~730 records per station,
# so we'll get everything in one request. Keeping pagination
# logic anyway in case the date range is extended later.
PAGE_SIZE = 1000

def fetch_station_data(
    station_id: str,
    start_date: str,
    end_date: str,
) -> list[dict]:
    """
    Fetch all daily summary records for a given station and
    date range from the NOAA Access Data Service API.

    Paginates automatically if the date range spans more than
    1000 days (the API's max records per request).

    Args:
        station_id:  Station identifier without prefix, e.g. "USW00013994"
        start_date:  ISO date string "YYYY-MM-DD"
        end_date:    ISO date string "YYYY-MM-DD"

    Returns:
        List of raw record dicts from the API.
    """
    all_records = []
    offset      = 1    # NOAA uses 1-based offset, not 0-based

    while True:
        params = {
            "dataset":   "daily-summaries",
            "stations":  station_id,
            "startDate": start_date,
            "endDate":   end_date,
            "datatype":  ",".join(DATA_TYPES),
            "format":    "json",
            "units":     "metric",
            "offset":    offset,
            "limit":     PAGE_SIZE,
        }

        logger.info(f"  Fetching {station_id} offset={offset}...")

        try:
            resp = requests.get(
                NOAA_BASE_URL,
                headers=NOAA_HEADERS,
                params=params,
                timeout=60
            )
        except requests.RequestException as e:
            logger.error(f"  Request error: {e}")
            break

        if resp.status_code != 200:
            logger.error(f"  HTTP {resp.status_code}: {resp.text[:200]}")
            break

        try:
            records = resp.json()
        except Exception as e:
            logger.error(f"  JSON parse error: {e}")
            break

        if not records:
            # Empty response means we've fetched everything
            break

        all_records.extend(records)
        logger.info(f"  Got {len(records)} records (total so far: {len(all_records)})")

        if len(records) < PAGE_SIZE:
            # Last page — fewer records than page size means no more data
            break

        offset += PAGE_SIZE

    return all_records


print("✓ fetch_station_data() defined")

## Cell 6 — Parse Weather Records

In [0]:
def parse_records(records: list[dict], source_label: str) -> pd.DataFrame | None:
    """
    Normalize raw NOAA API records into the canonical schema.

    Steps:
      1. Convert list of dicts to Pandas DataFrame
      2. Rename columns to canonical names
      3. Derive year and month from the DATE field
      4. Cast numeric fields to float
      5. Convert Celsius temps to Fahrenheit
         Formula: F = (C × 9/5) + 32
      6. Add provenance metadata

    Args:
        records:      Raw records from fetch_station_data()
        source_label: Label for the _source provenance column

    Returns:
        Normalized Pandas DataFrame or None if records is empty.
    """
    if not records:
        logger.warning(f"  No records to parse for {source_label}")
        return None

    pdf = pd.DataFrame(records)
    logger.info(f"  Raw shape: {pdf.shape[0]} rows x {pdf.shape[1]} cols")

    # ── Rename to canonical names ─────────────────────────────
    rename_map = {
        "STATION": "station_id",
        "DATE":    "date",
        "TMAX":    "tmax_c",    # note: renamed to tmax_c
        "TMIN":    "tmin_c",    # note: renamed to tmin_c
        "PRCP":    "prcp",
        "SNOW":    "snow",
        "SNWD":    "snwd",
        "AWND":    "awnd",
    }
    pdf = pdf.rename(columns=rename_map)

    # ── Derive year and month from date ───────────────────────
    pdf["year"]  = pd.to_datetime(pdf["date"]).dt.year
    pdf["month"] = pd.to_datetime(pdf["date"]).dt.month

    # ── Cast numeric fields to float ──────────────────────────
    # Do this BEFORE the Fahrenheit conversion so we're doing
    # math on actual numbers, not strings.
    numeric_cols = ["tmax_c", "tmin_c", "prcp", "snow", "snwd", "awnd"]
    for col in numeric_cols:
        if col in pdf.columns:
            pdf[col] = pd.to_numeric(pdf[col], errors="coerce")

    # ── Convert Celsius → Fahrenheit ─────────────────────────
    # F = (C × 9/5) + 32
    # Guard against stations that don't record temperature —
    # the Science Center station has fewer data types than Lambert.
    # If the source column doesn't exist, the Fahrenheit column
    # will be added as null in the "add missing columns" step below.
    if "tmax_c" in pdf.columns:
        pdf["tmax_f"] = (pdf["tmax_c"] * 9 / 5) + 32
    if "tmin_c" in pdf.columns:
        pdf["tmin_f"] = (pdf["tmin_c"] * 9 / 5) + 32

    # ── Add missing canonical columns as null ─────────────────
    for col in CANONICAL_SCHEMA:
        if col not in pdf.columns:
            pdf[col] = None

    # ── Keep only canonical columns ───────────────────────────
    pdf = pdf[[col for col in CANONICAL_SCHEMA if col in pdf.columns]]

    # ── Provenance ────────────────────────────────────────────
    pdf["_source"]      = source_label
    pdf["_ingested_at"] = datetime.now(timezone.utc).isoformat()

    logger.info(f"  ✓ Parsed {len(pdf):,} rows")
    return pdf


print("✓ parse_records() defined")

## Cell 7 — Main Ingestion & Write Parquet

In [0]:
print("Starting NOAA weather ingestion...")
print("=" * 60)

all_frames: list[pd.DataFrame] = []

for station_name, station_id in STATIONS.items():
    # Strip the GHCND: prefix if present — API wants bare ID
    bare_id = station_id.replace("GHCND:", "")
    label   = f"noaa_{station_name}"

    print(f"\n{'─' * 40}")
    print(f"Station: {station_name} ({bare_id})")

    records = fetch_station_data(bare_id, START_DATE, END_DATE)

    if not records:
        print(f"  ⚠ No records returned for {station_name}")
        continue

    pdf = parse_records(records, source_label=label)
    if pdf is None:
        continue

    all_frames.append(pdf)
    print(f"  ✓ {len(pdf):,} daily records")

if not all_frames:
    raise RuntimeError("No weather data was successfully ingested.")

# ── Combine both stations and convert to Spark ────────────────
combined_pdf = pd.concat(all_frames, ignore_index=True)

sdf = spark.createDataFrame(combined_pdf)

# Cast to canonical Spark types
for col_name, spark_type in CANONICAL_SCHEMA.items():
    if col_name in sdf.columns:
        sdf = sdf.withColumn(col_name, F.col(col_name).cast(spark_type))

# ── Write Parquet partitioned by year/month ───────────────────
# Same partition scheme as city crime data for clean joins later
sdf.write.mode("overwrite") \
   .partitionBy("year", "month") \
   .parquet(RAW_OUTPUT_PATH)

total_rows = sdf.count()
print(f"\n{'=' * 60}")
print(f"✓ Wrote {total_rows:,} rows → {RAW_OUTPUT_PATH}")
sdf.printSchema()

## Cell 8 — Quality Checks 

In [0]:
print("=== TEMPERATURE & PRECIPITATION SUMMARY ===")
display(
    sdf.select(
        # Celsius
        F.round(F.avg("tmax_c"), 2).alias("avg_tmax_c"),
        F.round(F.avg("tmin_c"), 2).alias("avg_tmin_c"),
        # Fahrenheit
        F.round(F.avg("tmax_f"), 2).alias("avg_tmax_f"),
        F.round(F.avg("tmin_f"), 2).alias("avg_tmin_f"),
        # Precipitation
        F.round(F.avg("prcp"), 2).alias("avg_prcp_mm"),
        F.round(F.max("prcp"), 2).alias("max_prcp_mm"),
        F.round(F.max("snow"), 2).alias("max_snow_mm"),
        F.sum("snow").alias("total_snow_mm"),
    )
)

## Cell 9 — Verify Parquet Write

In [0]:
persisted_df    = spark.read.parquet(RAW_OUTPUT_PATH)
persisted_count = persisted_df.count()

print(f"Rows read back from Parquet: {persisted_count:,}")

if persisted_count == total_rows:
    print("✓ Row count matches — write verified")
else:
    print(f"⚠ Mismatch: in-memory={total_rows:,}, on-disk={persisted_count:,}")